# Phase 4 — Retrieval-Augmented Generation for Pharmacovigilance Q&A

### FAERS Intelligence Platform

This phase turns the calibrated signal corpus into an interactive Q&A surface backed by
ChromaDB and Claude. The retrieval layer is designed for the specific failure modes that
plagued earlier naive RAG attempts.

### Problems with the earlier RAG

- `where={"drug":{"$contains":x}}` — in ChromaDB, `$contains` works only on
  `where_document`, not `where`. The filter silently did nothing, so a question about
  semaglutide could return an answer about amlodipine.
- Evaluation by **answer-text keyword match** is broken — a response saying
  "no rosiglitazone data" still contains 'rosiglitazone' and gets counted as correct.
- Only one severity label was surfaced, ignoring Phase 3's dual-severity finding
  (PT-level clinical risk vs drug-PT empirical outcome).

### Design improvements

1. **Exact metadata matching** — `where={"drug": drug_norm}` against the normalized key
   that Phase 3 used during indexing.
2. **Hybrid retrieval** — semantic search (embeddings) **and** metadata filters
   (drug / severity / group). Unscoped queries fall back to global search.
3. **Dual-severity surfacing** — both `pt_severity` (clinical) and `final_severity`
   (calibrated) are included in the answer context.
4. **Retrieval-stage evaluation** — precision is measured against the metadata of the
   **retrieved documents**, not against keyword matches in the answer text.
5. **Citation enforcement** — the answer must cite signal IDs, which downstream
   evaluation can verify.
6. **Per-drug split retrieval** — when multiple drugs are referenced, k is divided
   evenly across them so a sparse drug (e.g., rosiglitazone with 2 reports) is not
   starved by a common one (e.g., metformin with 625).
7. **UNCLASSIFIED hidden by default** — only included when the user explicitly asks
   ("show me unknown-severity signals"). Keeps results clinically meaningful.

> **Input:** `faers.db` and `ChromaDB(faers_signals_v6)` from Phase 3.
> **Output:** evaluation report and demo Q&A.

## 1. Setup & Load State

In [ ]:
# ============================================================================
# Section 1 — Setup & Load State (self-healing ChromaDB)
# ----------------------------------------------------------------------------
# A ChromaDB HNSW index produced by an older client version can fail to load in a
# newer version with "Error loading hnsw index". To make Phase 4 robust to version
# drift, we attempt to load the existing collection first; on any failure (or if the
# collection has fewer documents than `calib`), we rebuild it from scratch using
# `calib` + saved interpretations.
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')
!pip install -q anthropic chromadb scikit-learn

import os, json, sqlite3, time, hashlib
import pandas as pd, numpy as np
import chromadb
from anthropic import Anthropic
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings("ignore")

PROJECT_DIR = "/content/drive/MyDrive/FAERS_Intelligence"
DB_PATH     = os.path.join(PROJECT_DIR, "data", "db", "faers.db")
CHROMA_DIR  = os.path.join(PROJECT_DIR, "data", "chromadb")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)
CACHE_DIR   = os.path.join(PROJECT_DIR, "cache");   os.makedirs(CACHE_DIR,   exist_ok=True)

# Anthropic API key — prefer Colab secret, fall back to env var. Never hard-code.
try:
    from google.colab import userdata
    API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    API_KEY = os.environ.get('ANTHROPIC_API_KEY','')
assert API_KEY, "ANTHROPIC_API_KEY missing — set in Colab Secrets or env var"
client = Anthropic(api_key=API_KEY)

# Calibrated signals — needed for evaluation and for the rebuild path
calib = pd.read_csv(os.path.join(RESULTS_DIR, "severity_calibrated_v6.csv"))
print(f"Calibrated signals loaded: {len(calib):,}")

# Drug context (used in prompts and in rebuild documents)
DRUG_CONTEXT = {
    'ciprofloxacin':  ('fluoroquinolone antibiotic','bacterial infections'),
    'isotretinoin':   ('retinoid','severe acne'),
    'clozapine':      ('atypical antipsychotic','treatment-resistant schizophrenia'),
    'warfarin':       ('vitamin K antagonist','anticoagulation'),
    'dolutegravir':   ('integrase inhibitor (GSK)','HIV'),
    'cabotegravir':   ('long-acting integrase inhibitor (GSK)','HIV PrEP/treatment'),
    'mepolizumab':    ('anti-IL-5 monoclonal antibody (GSK)','severe eosinophilic asthma'),
    'semaglutide':    ('GLP-1 receptor agonist','type 2 diabetes, obesity'),
    'tirzepatide':    ('GLP-1/GIP dual agonist','type 2 diabetes, obesity'),
    'lecanemab':      ('anti-amyloid mAb','early Alzheimer disease'),
    'pembrolizumab':  ('anti-PD-1 checkpoint inhibitor','various cancers'),
    'metformin':      ('biguanide','type 2 diabetes'),
    'lisinopril':     ('ACE inhibitor','hypertension'),
    'omeprazole':     ('proton pump inhibitor','GERD'),
    'amoxicillin':    ('penicillin antibiotic','bacterial infections'),
    'amlodipine':     ('calcium channel blocker','hypertension'),
    'rofecoxib':      ('COX-2 inhibitor (withdrawn)','pain'),
    'rosiglitazone':  ('thiazolidinedione (restricted)','type 2 diabetes'),
}
DRUGS = sorted(set(DRUG_CONTEXT) & set(calib['target_drug'].dropna().unique()))
print(f"Drugs in calib & context: {len(DRUGS)}")

# ---- ChromaDB: load if available, otherwise rebuild from calib + interpretations ----
def _build_collection(chroma_client):
    """Rebuild the ChromaDB collection from calib and saved interpretations."""
    try: chroma_client.delete_collection("faers_signals_v6")
    except Exception: pass
    coll = chroma_client.create_collection(name="faers_signals_v6",
                                           metadata={"hnsw:space": "cosine"})
    docs, metas, ids = [], [], []
    for idx, row in tqdm(calib.iterrows(), total=len(calib), desc="Re-indexing"):
        drug_class, indication = DRUG_CONTEXT.get(row['target_drug'], ('unknown','unknown'))
        doc = (f"Drug: {row['target_drug']} ({drug_class}). Indication: {indication}. "
               f"Group: {row['target_group']}. Role: {row['target_role']}. "
               f"Adverse Event: {row['pt']}. "
               f"Reports: {int(row['a'])}. Expected: {row['expected']:.1f}. "
               f"PRR: {row['prr']:.2f}. EBGM: {row['ebgm']:.2f} (EB05: {row['eb05']:.2f}). "
               f"Signal: {row['signal_strength']}. Calibrated severity: {row['final_severity']} "
               f"(source: {row['calib_source']}). ")
        if pd.notna(row['n_cases']) and row['n_cases'] >= 5:
            doc += (f"Actual cases: n={int(row['n_cases'])}, death rate {row['death_rate']*100:.0f}%, "
                    f"serious rate {row['serious_rate']*100:.0f}%. ")
        docs.append(doc)
        metas.append({
            'drug': str(row['target_drug']), 'event': str(row['pt']),
            'group': str(row['target_group']), 'role': str(row['target_role']),
            'drug_class': drug_class, 'ebgm': float(row['ebgm']),
            'signal_strength': str(row['signal_strength']),
            'final_severity': str(row['final_severity']),
            'n_cases': int(row['n_cases']) if pd.notna(row['n_cases']) else 0,
            'death_rate': float(row['death_rate']) if pd.notna(row['death_rate']) else 0.0,
        })
        ids.append(f"sig_{idx}")
    # Drug interpretations — load saved JSON, flatten dict-schema to clean prose
    interp_path = os.path.join(RESULTS_DIR, "drug_interpretations_v6.json")
    SECTION_LABELS = {
        "overall_safety_profile":  "Overall safety profile",
        "most_concerning_signals": "Most concerning signals",
        "expected_vs_novel":       "Expected vs novel signals",
        "class_comparison":        "Class comparison",
        "risk_benefit":            "Risk-benefit",
        "recommendations":         "Recommendations",
    }
    def _flatten_interp(interp):
        # New schema (dict) -> clean prose. Old schema (str) -> pass through.
        if isinstance(interp, dict):
            parts = []
            for k, label in SECTION_LABELS.items():
                v = interp.get(k)
                if isinstance(v, str) and v.strip():
                    parts.append(f"{label}: {v.strip()}")
            return " ".join(parts)
        return str(interp) if interp is not None else ""
    if os.path.exists(interp_path):
        with open(interp_path, encoding="utf-8") as f:
            interpretations = json.load(f)
        for drug, interp in interpretations.items():
            drug_class, indication = DRUG_CONTEXT.get(drug, ('unknown','unknown'))
            interp_text = _flatten_interp(interp)
            if not interp_text:
                continue
            docs.append(f"Drug safety assessment for {drug} ({drug_class}, {indication}): {interp_text}")
            metas.append({'drug':drug,'event':'INTERPRETATION','group':'meta','role':'meta',
                          'drug_class':drug_class,'ebgm':0,'signal_strength':'N/A',
                          'final_severity':'N/A','n_cases':0,'death_rate':0})
            ids.append(f"interp_{drug}")
    B = 4000
    for i in range(0, len(docs), B):
        coll.add(documents=docs[i:i+B], metadatas=metas[i:i+B], ids=ids[i:i+B])
    return coll

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    collection = chroma_client.get_collection("faers_signals_v6")
    n = collection.count()                      # Attempt to read the index — except if it fails
    if n < len(calib):                          # Fewer docs than signals -> stale, rebuild
        print(f"Collection has {n} docs but calib has {len(calib)} — rebuilding for freshness.")
        collection = _build_collection(chroma_client)
    else:
        print(f"ChromaDB collection loaded — {n:,} documents")
except Exception as e:
    print(f"Could not load existing index ({type(e).__name__}). Rebuilding from calib...")
    collection = _build_collection(chroma_client)

print(f"ChromaDB collection 'faers_signals_v6' — {collection.count():,} documents")

## 2. Query Parsing — extract structured filters from natural language

An LLM extracts structured metadata filters (drug, severity, group) from the user's
question. This is the root-cause fix for the v5 bug where the drug filter silently
did nothing.

In [2]:
# === v6.1: parser now extracts `drugs: [list]` (supports multi-drug comparison) ===
# === v6.5: also detects `include_unclassified` so UNCLASSIFIED is hidden by default ===
PARSER_SYS = """Extract structured filters from a pharmacovigilance question.
Return ONLY a JSON object with these keys (omit a key if not mentioned):
  - drugs: list of lowercase drug names from the allowed list (1+ drugs). Use [] if none mentioned. For "A vs B", return both.
  - severity: one of FATAL, SEVERE, MODERATE, MILD, or null
  - group: one of validation, gsk, trending, negative_control, or null
  - top_k: integer 1-20 (default 5; use more for comparison queries)
  - intent: one of {risk_profile, severity_focus, comparison, mechanism, signal_list, general}
  - include_unclassified: true ONLY if the user explicitly asks for unclassified / unlabeled / unknown-severity signals; otherwise false

For comparison queries (intent=\'comparison\'), put ALL drugs being compared in `drugs` AND use top_k>=10 so each drug gets representation.

Allowed drugs: """ + ", ".join(DRUGS)

def parse_query(q: str) -> dict:
    try:
        resp = client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=300,
            system=PARSER_SYS,
            messages=[{"role":"user","content":f"Question: {q}\nReturn JSON only."}])
        txt = resp.content[0].text.strip()
        if txt.startswith("```"):
            txt = txt.split("\n",1)[1].rsplit("```",1)[0]
        f = json.loads(txt)
    except Exception:
        f = {"intent":"general"}

    # backward-compat: accept old `drug: str` and migrate to `drugs: [str]`
    if "drug" in f and "drugs" not in f:
        f["drugs"] = [f.pop("drug")] if f["drug"] else []
    f.setdefault("drugs", [])
    f.setdefault("top_k", 5)
    f.setdefault("include_unclassified", False)

    # sanitize
    f["drugs"] = [d.lower() for d in f["drugs"] if isinstance(d,str) and d.lower() in DRUGS]
    if f.get("severity") not in {None,"FATAL","SEVERE","MODERATE","MILD"}: f["severity"]=None
    if f.get("group") not in {None,"validation","gsk","trending","negative_control"}: f["group"]=None
    f["include_unclassified"] = bool(f.get("include_unclassified", False))

    # auto-boost top_k for comparison so each drug gets coverage
    if f.get("intent")=="comparison" and len(f["drugs"])>=2:
        f["top_k"] = max(f["top_k"], 5*len(f["drugs"]))
    return f

# Smoke test
for q in ["What are the fatal signals for semaglutide?",
         "Compare rosiglitazone and metformin cardiac risk",
         "Which GSK drugs have the strongest signals?",
         "Tell me about ARIA-E",
         "Show me unclassified / unknown-severity signals for warfarin",
         "Compare warfarin vs lisinopril vs amlodipine bleeding risk"]:
    print(q, "→", parse_query(q))


What are the fatal signals for semaglutide? → {'drugs': ['semaglutide'], 'severity': 'FATAL', 'top_k': 5, 'intent': 'severity_focus', 'include_unclassified': False}
Compare rosiglitazone and metformin cardiac risk → {'drugs': ['rosiglitazone', 'metformin'], 'severity': None, 'group': None, 'top_k': 10, 'intent': 'comparison', 'include_unclassified': False}
Which GSK drugs have the strongest signals? → {'drugs': [], 'group': 'gsk', 'top_k': 10, 'intent': 'signal_list', 'include_unclassified': False}
Tell me about ARIA-E → {'drugs': ['lecanemab'], 'severity': None, 'group': None, 'top_k': 5, 'intent': 'mechanism', 'include_unclassified': False}
Show me unclassified / unknown-severity signals for warfarin → {'drugs': ['warfarin'], 'severity': None, 'group': None, 'top_k': 5, 'intent': 'signal_list', 'include_unclassified': True}
Compare warfarin vs lisinopril vs amlodipine bleeding risk → {'drugs': ['warfarin', 'lisinopril', 'amlodipine'], 'severity': None, 'group': None, 'top_k': 15, 'in

## 3. Hybrid Retrieval (semantic + metadata filter)

ChromaDB's `where` clause only supports exact matching reliably, so we apply structured
filters via `where` and let semantic similarity handle the natural-language part.

UNCLASSIFIED signals are excluded by default so the model focuses on labeled signals.
Users can opt in explicitly via the `include_unclassified` flag in the parsed query.

In [ ]:
def _query_one(query, where, k):
    """Single ChromaDB query with fallback. Returns normalized result dict."""
    try:
        r = collection.query(query_texts=[query], n_results=k, where=where)
        return r, False
    except Exception:
        r = collection.query(query_texts=[query], n_results=k)
        return r, True

def retrieve(query: str, filt: dict, k: int = 5):
    """Multi-drug-aware retrieval.

    Strategy:
      - 0 or 1 drug -> single ChromaDB call.
      - 2+ drugs    -> per-drug calls, each capped at k_per_drug = max(3, k // len(drugs)),
                       then merge. Guarantees each requested drug gets coverage even when
                       one is heavily underrepresented in the corpus.

    UNCLASSIFIED signals are excluded by default. They are only included when
    filt['include_unclassified'] is True; when the user filters by a specific severity,
    UNCLASSIFIED is naturally excluded.
    """
    sev = filt.get("severity"); grp = filt.get("group")
    drugs = filt.get("drugs") or []
    include_unclass = bool(filt.get("include_unclassified", False))

    def _aux_conds():
        a = []
        if sev:
            a.append({"final_severity": {"$eq": sev}})
        elif not include_unclass:
            # Default: exclude UNCLASSIFIED. Interpretation docs use final_severity='N/A' so they pass through.
            a.append({"final_severity": {"$ne": "UNCLASSIFIED"}})
        if grp: a.append({"group": {"$eq": grp}})
        return a

    if len(drugs) <= 1:
        conds = _aux_conds()
        if len(drugs) == 1: conds.append({"drug": {"$eq": drugs[0]}})
        where = None
        if len(conds) == 1: where = conds[0]
        elif len(conds) > 1: where = {"$and": conds}
        res, fb = _query_one(query, where, k)
        res["_fallback"] = fb
        return res

    # Multi-drug: per-drug retrieval, merged
    k_each = max(3, k // len(drugs))
    docs, metas, ids, dists = [], [], [], []
    fallback_any = False
    for d in drugs:
        conds = [{"drug": {"$eq": d}}] + _aux_conds()
        where = conds[0] if len(conds)==1 else {"$and": conds}
        r, fb = _query_one(query, where, k_each)
        fallback_any = fallback_any or fb
        if r.get("documents") and r["documents"][0]:
            docs.extend(r["documents"][0])
            metas.extend(r["metadatas"][0])
            ids.extend(r["ids"][0])
            if r.get("distances") and r["distances"][0]:
                dists.extend(r["distances"][0])

    # Rebuild ChromaDB-shaped response
    merged = {
        "documents":  [docs],
        "metadatas":  [metas],
        "ids":        [ids],
        "distances":  [dists] if dists else None,
        "_fallback":  fallback_any,
    }
    return merged

# Tests
r = retrieve("cardiac risk", {"drugs":["rosiglitazone","metformin"]}, k=10)
drugs_back = [m.get("drug") for m in r["metadatas"][0]]
sev_back = [m.get("final_severity") for m in r["metadatas"][0]]
print(f"multi-drug rosi+met: {len(drugs_back)} hits -> "
      f"rosiglitazone={drugs_back.count('rosiglitazone')}, metformin={drugs_back.count('metformin')}")
print(f"  UNCLASSIFIED in results: {sev_back.count('UNCLASSIFIED')} (should be 0)")

r = retrieve("fatal signals", {"drugs":["semaglutide"],"severity":"FATAL"}, k=3)
print(f"single-drug + sev  : {len(r['metadatas'][0])} hits, "
      f"all FATAL? {all(m.get('final_severity')=='FATAL' for m in r['metadatas'][0])}")

# Verify the opt-in flag works
r = retrieve("any signals", {"drugs":["warfarin"], "include_unclassified": True}, k=10)
sev_back = [m.get("final_severity") for m in r["metadatas"][0]]
print(f"opt-in unclassified: {sev_back.count('UNCLASSIFIED')} UNCLASSIFIED in {len(sev_back)} hits (may be >0)")

## 4. Answer Synthesis — grounded, cited, dual-severity aware

The model answers using only the retrieved context, cites each claim with `[id]`, and
surfaces both clinical and empirical severity perspectives from Phase 3.

In [4]:
# Clinically benign PT list — if death_rate >> 0 on these, it's a patient-population signal,
# not a drug-induced fatal mechanism. LLM is instructed to flag these explicitly.
BENIGN_PTS = {
    "TASTE DISORDER","DYSGEUSIA","HYPOAESTHESIA","PARAESTHESIA","MIGRAINE","HEADACHE",
    "NAUSEA","FATIGUE","DIZZINESS","DRY MOUTH","ARTHRALGIA","PRURITUS","RASH",
    "DRUG INEFFECTIVE","OFF LABEL USE","PRODUCT DOSE OMISSION ISSUE","BACK PAIN",
    "COUGH","CONSTIPATION","DIARRHOEA","ABDOMINAL PAIN","ASTHENIA","INSOMNIA",
    "ALOPECIA","DRY SKIN","EYE IRRITATION","VOMITING","DECREASED APPETITE"
}

def _is_pop_effect(m):
    """High death_rate on clinically benign PT → likely patient-population effect."""
    try:
        dr = float(m.get("death_rate", 0) or 0)
        n  = int(m.get("n_cases", 0) or 0)
        ev = str(m.get("event","")).upper()
        return (dr >= 0.30) and (n >= 10) and any(b in ev for b in BENIGN_PTS)
    except Exception:
        return False

ANSWER_SYS = """You are a clinical pharmacovigilance assistant grounded ONLY in the provided FAERS evidence.

RULES (strict):
1. Use ONLY facts from the CONTEXT block. If insufficient, say "Insufficient data in retrieved context."
2. Every factual claim ends with a citation like [sig_1234]. Never invent citation IDs.
3. When discussing severity, surface BOTH:
   - "empirical outcome severity" = `final_severity` (FAERS observed mortality/serious outcomes)
   - "clinical PT risk" = the inherent severity of the MedDRA term itself
   If they disagree, point it out explicitly.
4. **Small-sample caveat (mandatory)**: when citing `death_rate` or `serious_rate`, ALWAYS state n alongside. If n<10, append "(small-n, unstable estimate)" right after the rate. Do NOT highlight high mortality rates from n<10 as headline findings.
5. **Patient-population effect warning (mandatory)**: when a CONTEXT row is marked `pop_effect=1`, the high death_rate is likely from co-reported severe events in the same case (e.g. mepolizumab patients are severely asthmatic), NOT from the PT itself. Append "(likely patient-population effect — PT not typically fatal, deaths likely from co-occurring SAEs in the same report)" to that claim. Do NOT lead the answer with such PTs as "fatal signals".
6. **Comparison queries**: if multiple drugs are present in CONTEXT, structure the answer as a side-by-side comparison per drug. If one drug has very few hits, state that explicitly (e.g. "rosiglitazone has limited representation in current FAERS quarters — only N signals retrieved").
7. Use precise numbers (a, EBGM, death_rate) from context — never invent.
8. End with one-line FAERS limitation caveat (reporting bias, no denominator, no causality).

Format:
- Direct answer (2-4 sentences)
- Key signals (bullet list; mark small-n and pop-effect where applicable)
- Caveat
"""

def synth_answer(query: str, retrieved: dict) -> dict:
    docs  = retrieved['documents'][0]
    metas = retrieved['metadatas'][0]
    ids   = retrieved['ids'][0]
    if not docs:
        return dict(answer="Insufficient data in retrieved context.", citations=[], retrieved_n=0)

    ctx_lines = []
    pop_flags = []
    for i, m in enumerate(metas):
        pop = _is_pop_effect(m)
        pop_flags.append(pop)
        ctx_lines.append(
            f"[{ids[i]}] (drug={m.get('drug')}, event={m.get('event')}, "
            f"severity={m.get('final_severity')}, EBGM={m.get('ebgm'):.1f}, "
            f"n_cases={m.get('n_cases')}, death_rate={m.get('death_rate'):.2f}, "
            f"small_n={int(m.get('n_cases',0)<10)}, pop_effect={int(pop)})\n{docs[i]}"
        )
    ctx = "\n\n".join(ctx_lines)

    # tell user (in metadata of return) how many pop-effect rows were flagged
    user = f"QUESTION:\n{query}\n\nCONTEXT:\n{ctx}"
    resp = client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=1200,
        system=ANSWER_SYS,
        messages=[{"role":"user","content":user}])
    text = resp.content[0].text
    cites = [s.strip("[]") for s in __import__('re').findall(r"\[(sig_\d+|interp_[a-z]+)\]", text)]
    return dict(answer=text, citations=cites, retrieved_n=len(docs),
                retrieved_ids=list(ids), retrieved_meta=list(metas),
                pop_effect_flagged=sum(pop_flags))


## 5. End-to-End Q&A (with cache)

In [5]:
import re as _re
def _key(q): return hashlib.sha1(q.lower().strip().encode()).hexdigest()[:16]

def ask(query: str, top_k: int = 5, use_cache: bool = True, verbose: bool = False) -> dict:
    cache_f = os.path.join(CACHE_DIR, f"qa_{_key(query)}.json")
    if use_cache and os.path.exists(cache_f):
        return json.load(open(cache_f))

    filt = parse_query(query)
    filt["top_k"] = filt.get("top_k") or top_k
    retrieved = retrieve(query, filt, k=filt["top_k"])
    out = synth_answer(query, retrieved)
    out["query"] = query; out["filters"] = filt
    out["retrieval_fallback"] = bool(retrieved.get("_fallback", False))
    json.dump(out, open(cache_f,"w"), default=str)
    if verbose:
        print(f"Filters: {filt}")
        print(f"Retrieved: {out['retrieved_n']} docs (fallback={out['retrieval_fallback']})")
    return out

# v6.1: invalidate stale cache since prompts changed
import glob
for f in glob.glob(os.path.join(CACHE_DIR,"qa_*.json")):
    os.remove(f)
print("Cleared QA cache (prompts changed in v6.1)")

# Demo
demo = ask("Compare rosiglitazone and metformin cardiac risk", verbose=True)
print(demo["answer"][:800])


Cleared QA cache (prompts changed in v6.1)
Filters: {'drugs': ['rosiglitazone', 'metformin'], 'severity': None, 'group': None, 'top_k': 10, 'intent': 'comparison', 'include_unclassified': False}
Retrieved: 8 docs (fallback=False)
**Direct Comparison**: Rosiglitazone shows limited cardiac signals in the current FAERS data (only 2 total signals, both neurological), while metformin demonstrates multiple cardiac-related safety signals. Rosiglitazone has very limited representation with only 10 total cases across 2 quarters, compared to metformin's 157 cardiac-related cases, making direct cardiac risk comparison difficult.

**Key Cardiac Signals by Drug**:

**Rosiglitazone**: 
- No direct cardiac signals detected in retrieved context
- Limited overall representation (10 total cases) (small-n, unstable estimates)
- Signals focus on neurological toxicity rather than cardiovascular events

**Metformin**:
- **Arteriosclerosis coronary artery**: EBGM=25.9, n=99, death rate 31% [sig_2008]
- **Sin

## 6. Honest Retrieval-Stage Evaluation

Keyword matching on the answer text is unreliable — a response stating "no rosiglitazone
data" still contains the word "rosiglitazone" and would score as correct. We evaluate on
the **retrieved documents' metadata** instead:

- **drug_precision** — fraction of retrieved hits whose drug matches the query's drug
- **severity_precision** — analogous for severity
- **on-topic recall** — how often the ground-truth (drug, event) appears in the retrieved set

In [6]:
# Build evaluation set
def make_eval_set(n_per_type: int = 5, seed: int = 42):
    rng = np.random.RandomState(seed)
    items = []
    for d in DRUGS[:n_per_type]:
        items.append(dict(qtype="drug_risk", drug=d,
            query=f"What are the safety signals for {d}?"))
    for d in DRUGS[:n_per_type]:
        fatal = calib[(calib.target_drug==d)&(calib.final_severity=="FATAL")]
        if len(fatal):
            items.append(dict(qtype="drug_fatal", drug=d, severity="FATAL",
                query=f"What are the fatal adverse events reported for {d}?"))
    sample = (calib.sort_values("ebgm",ascending=False)
                   .groupby("target_drug").head(1).head(n_per_type))
    for _,r in sample.iterrows():
        items.append(dict(qtype="drug_pt", drug=r.target_drug, pt=r.pt,
            query=f"Is {r.pt} a known signal for {r.target_drug}?"))
    drug_pairs = [("rosiglitazone","metformin"), ("warfarin","lisinopril"),
                  ("semaglutide","tirzepatide")]
    for d1,d2 in drug_pairs:
        if d1 in DRUGS and d2 in DRUGS:
            items.append(dict(qtype="comparison", drugs=[d1,d2],
                query=f"Compare {d1} and {d2} safety signals"))
    return items

eval_set = make_eval_set(n_per_type=5)
print(f"Eval set: {len(eval_set)} queries  "
      f"(incl. {sum(1 for i in eval_set if i['qtype']=='comparison')} comparisons)")

def eval_one(item):
    q = item["query"]
    filt = parse_query(q)
    retrieved = retrieve(q, filt, k=max(filt.get("top_k",5),5))
    metas = retrieved["metadatas"][0] if retrieved["metadatas"] else []
    drugs_hit = [m.get("drug") for m in metas]
    sev_hit   = [m.get("final_severity") for m in metas]
    evt_hit   = [m.get("event") for m in metas]

    res = dict(qtype=item["qtype"], query=q, n_retrieved=len(metas), filters=filt)
    if item.get("drug"):
        res["drug_precision"] = (sum(d==item["drug"] for d in drugs_hit)/len(metas)) if metas else 0.0
    if item.get("drugs"):
        target = set(item["drugs"])
        res["drug_precision"] = (sum(d in target for d in drugs_hit)/len(metas)) if metas else 0.0
        # NEW (v6.2): per-drug min-1 coverage assertion
        res["all_drugs_covered"] = int(target.issubset(set(drugs_hit)))
        for d in target:
            res[f"hits_{d}"] = drugs_hit.count(d)
    if item.get("severity"):
        res["severity_precision"] = (sum(s==item["severity"] for s in sev_hit)/len(metas)) if metas else 0.0
    if item.get("pt"):
        res["pt_in_top5"] = int(any(e==item["pt"] for e in evt_hit))
    return res

rows = [eval_one(it) for it in tqdm(eval_set, desc="Retrieval eval")]
ev = pd.DataFrame(rows)

print("\n=== Retrieval Precision (metadata-based) ===")
for col in ["drug_precision","severity_precision","pt_in_top5","all_drugs_covered"]:
    if col in ev.columns:
        s = ev[col].dropna()
        if len(s): print(f"  {col:<22s}: {s.mean():.2%}  (n={len(s)})")

# Show per-drug coverage breakdown for comparisons (v6.2 transparency)
cmp_rows = ev[ev.qtype=="comparison"]
if len(cmp_rows):
    print("\n--- Per-drug hits in comparison queries ---")
    hit_cols = [c for c in cmp_rows.columns if c.startswith("hits_")]
    print(cmp_rows[["query"]+hit_cols].to_string(index=False))


Eval set: 17 queries  (incl. 3 comparisons)


Retrieval eval:   0%|          | 0/17 [00:00<?, ?it/s]


=== Retrieval Precision (metadata-based) ===
  drug_precision        : 100.00%  (n=17)
  severity_precision    : 100.00%  (n=4)
  pt_in_top5            : 100.00%  (n=5)
  all_drugs_covered     : 100.00%  (n=3)

--- Per-drug hits in comparison queries ---
                                             query  hits_rosiglitazone  hits_metformin  hits_lisinopril  hits_warfarin  hits_semaglutide  hits_tirzepatide
Compare rosiglitazone and metformin safety signals                 3.0             5.0              NaN            NaN               NaN               NaN
    Compare warfarin and lisinopril safety signals                 NaN             NaN              5.0            5.0               NaN               NaN
Compare semaglutide and tirzepatide safety signals                 NaN             NaN              NaN            NaN               5.0               5.0


## 7. Comparative Quality vs Naive RAG (v5-style)

Re-creates v5's buggy `$contains` filter intentionally so the two retrievers can be
compared in a single table.

In [7]:
def retrieve_naive_v5(query, drug=None, k=5):
    """v5 bug reproduction: $contains on metadata (broken)."""
    where = None
    if drug:
        where = {"drug": {"$contains": drug}}
    try:
        return collection.query(query_texts=[query], n_results=k, where=where)
    except Exception:
        return collection.query(query_texts=[query], n_results=k)

cmp_rows = []
for it in eval_set:
    if not it.get("drug"): continue
    v6 = retrieve(it["query"], parse_query(it["query"]), k=5)
    v5 = retrieve_naive_v5(it["query"], drug=it["drug"], k=5)
    p6 = np.mean([m.get("drug")==it["drug"] for m in v6["metadatas"][0]]) if v6["metadatas"][0] else 0
    p5 = np.mean([m.get("drug")==it["drug"] for m in v5["metadatas"][0]]) if v5["metadatas"][0] else 0
    cmp_rows.append(dict(query=it["query"][:60], v5_precision=p5, v6_precision=p6))
cmp = pd.DataFrame(cmp_rows)
print(cmp.to_string(index=False))
print(f"\nMean drug precision  — v5(naive): {cmp.v5_precision.mean():.2%}  |  v6(filtered): {cmp.v6_precision.mean():.2%}")

                                                       query  v5_precision  v6_precision
                 What are the safety signals for amlodipine?             0           1.0
                What are the safety signals for amoxicillin?             0           1.0
               What are the safety signals for cabotegravir?             0           1.0
              What are the safety signals for ciprofloxacin?             0           1.0
                  What are the safety signals for clozapine?             0           1.0
  What are the fatal adverse events reported for amlodipine?             0           1.0
 What are the fatal adverse events reported for amoxicillin?             0           1.0
What are the fatal adverse events reported for ciprofloxacin             0           1.0
   What are the fatal adverse events reported for clozapine?             0           1.0
Is AMYLOID RELATED IMAGING ABNORMALITY-OEDEMA/EFFUSION a kno             0           1.0
               Is APP

## 8. Demonstration Questions (showcase)

In [8]:
SHOWCASE = [
    "What are the strongest fatal signals for semaglutide?",
    "Compare cardiac risk: rosiglitazone vs metformin",
    "Which GSK portfolio drugs have the most severe signals?",
    "Does ciprofloxacin still show tendon-related signals in 2024?",
    "Tell me about ARIA-E for lecanemab",
    "Any unexpected dermatology signals in the negative-control group?",
]

results = []
for q in SHOWCASE:
    r = ask(q, top_k=6, verbose=False)
    results.append(r)
    print("="*78)
    print(f"Q: {q}")
    print(f"  filters: {r['filters']}")
    print(f"  retrieved: {r['retrieved_n']} (fallback={r.get('retrieval_fallback')})")
    print(f"  citations: {len(r['citations'])}")
    print()
    print(r["answer"][:700])
    print()

Q: What are the strongest fatal signals for semaglutide?
  filters: {'drugs': ['semaglutide'], 'severity': 'FATAL', 'top_k': 5, 'intent': 'severity_focus', 'include_unclassified': False}
  retrieved: 5 (fallback=False)
  citations: 7

The strongest fatal signals for semaglutide are medullary thyroid cancer and adenocarcinoma pancreas, both showing very high EBGM values despite zero or low observed death rates in FAERS [sig_3227][sig_3164].

**Key fatal signals for semaglutide:**
• **Medullary thyroid cancer** - EBGM 35.2, n=11, death rate 0% (small-n, unstable estimate) [sig_3227]
• **Adenocarcinoma pancreas** - EBGM 20.4, n=12, death rate 25% (small-n, unstable estimate) [sig_3164]  
• **Anaplastic thyroid cancer** - EBGM 15.2, n=3, death rate 0% (small-n, unstable estimate) [sig_3166]
• **Necrotizing pancreatitis** - EBGM 11.9, n=22, death rate 32% [sig_3239]
• **Pancreatic carcinoma** - EBGM 8.5, n=75, death rate 21% [

Q: Compare cardiac risk: rosiglitazone vs metformin
  filters: 

## 9. Save Artifacts

In [9]:
import json as _json
out_path = os.path.join(RESULTS_DIR, "rag_qa_v6.json")
_json.dump([{
    "query": r["query"], "filters": r["filters"], "answer": r["answer"],
    "citations": r["citations"], "retrieved_n": r["retrieved_n"],
    "retrieval_fallback": r.get("retrieval_fallback", False),
} for r in results], open(out_path,"w"), indent=2, default=str)
print("Saved:", out_path)

# Retrieval evaluation report
rep_path = os.path.join(RESULTS_DIR, "rag_retrieval_eval_v6.csv")
ev.to_csv(rep_path, index=False)
cmp_path = os.path.join(RESULTS_DIR, "rag_v5_vs_v6_comparison.csv")
cmp.to_csv(cmp_path, index=False)
print("Saved:", rep_path)
print("Saved:", cmp_path)

Saved: /content/drive/MyDrive/FAERS_Intelligence/results/rag_qa_v6.json
Saved: /content/drive/MyDrive/FAERS_Intelligence/results/rag_retrieval_eval_v6.csv
Saved: /content/drive/MyDrive/FAERS_Intelligence/results/rag_v5_vs_v6_comparison.csv


## 10. Phase 4 Complete

**Outputs**
- `rag_qa_v6.json` — demo Q&A (with citations, filters, metadata)
- `rag_retrieval_eval_v6.csv` — retrieval-stage evaluation (drug / severity / PT precision)
- `rag_v5_vs_v6_comparison.csv` — quantitative comparison against the v5 buggy RAG

**Key improvements**
- Exact metadata matching fixes v5's silent drug-filter bug.
- Natural language is auto-parsed into structured filters (`parse_query`).
- Retrieval-stage evaluation eliminates answer-keyword false positives.
- Phase 3's dual-severity finding is surfaced in every answer.

**Next: Phase 5** — Streamlit dashboard that exposes both severity perspectives, the
RAG Q&A panel, and the validation results.

In [10]:
print("="*70)
print("Phase 4 (v6) complete — Enhanced RAG")
print("="*70)
print(f"  Indexed documents : {collection.count():,}")
print(f"  Eval queries      : {len(eval_set)}")
print(f"  v6 drug precision : {cmp.v6_precision.mean():.1%}  (vs v5 naive: {cmp.v5_precision.mean():.1%})")
print(f"  Showcase answered : {len(results)}")
print("\nNext: Phase 5 v6 — dashboard surfacing dual-severity + RAG.")

Phase 4 (v6) complete — Enhanced RAG
  Indexed documents : 3,501
  Eval queries      : 17
  v6 drug precision : 100.0%  (vs v5 naive: 0.0%)
  Showcase answered : 6

Next: Phase 5 v6 — dashboard surfacing dual-severity + RAG.
